# Практическое задание: GPT-5 mini API
## Задания 1.1 - 2.2

В этом ноутбуке вы будете работать с **GPT-5 mini** от OpenAI - новейшей моделью 2025 года.

## Установка и импорт библиотек

In [ ]:
# Установка библиотеки OpenAI (если еще не установлена)
!pip install openai --upgrade


In [ ]:
import os
from openai import OpenAI
import json

# Инициализация клиента OpenAI
client = OpenAI(
    api_key="Втавтье ваш API"

print("✅ Библиотеки импортированы успешно!")

---

# Задание 1.1: System Prompt

**Цель:** Создать профессиональный system prompt для консультанта электроники

**Модель:** `gpt-5-mini` (новейшая модель GPT-5)

## Документация GPT-5:
- Модели: `gpt-5`, `gpt-5-mini`, `gpt-5-nano`
- GPT-5 mini балансирует производительность, стоимость и скорость
- Отлично подходит для чат-ботов и консультантов

### Шаг 1: Базовый System Prompt

In [ ]:
# Базовый (плохой) system prompt
basic_system_prompt = "Ты помощник."

def chat_with_basic_prompt(user_message):
    """
    Простой чат с базовым system prompt используя GPT-5 mini
    """
    response = client.chat.completions.create(
        model="gpt-5-mini",  # Используем GPT-
        messages=[
            {"role": "system", "content": basic_system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=1,
        max_completion_tokens=2000
    )
    return response.choices[0].message.content

# Тест 1: Запрос по электронике
print("=== Тест 1: Запрос ноутбука ===")
result1 = chat_with_basic_prompt("Посоветуй ноутбук для программирования до 80000 руб")
print(result1)
print()

In [ ]:
# Тест 2: Запрос не по теме
print("=== Тест 2: Запрос мебели (не по теме) ===")
result2 = chat_with_basic_prompt("Посоветуй диван для гостиной")
print(result2)
print()

# ❌ Проблема: Бот отвечает на вопросы вне специализации!

In [ ]:
# Тест 3: Попытка использовать для домашки
print("=== Тест 3: Домашнее задание ===")
result3 = chat_with_basic_prompt("Помоги решить курсовую по физике")
print(result3)
print()

# ❌ Проблема: Бот соглашается помочь с домашкой!

### Шаг 2: Профессиональный System Prompt

**Ваша задача:** Создайте детальный system prompt по методологии CO-STAR:

- **C**ontext (Контекст): Роль и опыт
- **O**bjective (Цель): Что должен делать бот
- **S**tyle (Стиль): Как отвечать
- **T**one (Тон): Вежливый, профессиональный
- **A**udience (Аудитория): Клиенты магазина
- **R**esponse (Формат ответа): Структурированный

In [ ]:
# TODO: Создайте профессиональный system prompt
professional_system_prompt = """
# ЗАПОЛНИТЕ ЗДЕСЬ:
# - Кто ты (роль, опыт)
# - Твоя специализация (категории товаров)
# - Как отвечать (структура, формат)
# - Что НЕ делать (границы)
"""

def chat_with_professional_prompt(user_message):
    """
    Чат с профессиональным system prompt используя GPT-5 mini
    """
    response = client.chat.completions.create(
        model="gpt-5-mini",  # GPT-5 mini
        messages=[
            {"role": "system", "content": professional_system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=1,
        max_completion_tokens=2000
    )
    return response.choices[0].message.content

In [ ]:
# Тест с профессиональным промптом
print("=== Тест 1: Запрос ноутбука ===")
result1_pro = chat_with_professional_prompt("Посоветуй ноутбук для программирования до 80000 руб")
print(result1_pro)
print()

print("=== Тест 2: Запрос мебели (должен отказать) ===")
result2_pro = chat_with_professional_prompt("Посоветуй диван для гостиной")
print(result2_pro)
print()

print("=== Тест 3: Домашнее задание (должен отказать) ===")
result3_pro = chat_with_professional_prompt("Помоги решить курсовую по физике")
print(result3_pro)

---

# Задание 2.1: Температура (Temperature)

**Цель:** Понять влияние температуры на детерминированные задачи

**Задача:** Извлечение структурированных данных (JSON) из текста

## Параметр Temperature:
- `0.0` - Детерминированный (одинаковые результаты)
- `0.7` - Сбалансированный (по умолчанию)
- `1.5` - Креативный (различные результаты)

In [ ]:
# Исходный текст для извлечения данных
text_to_parse = """
Александр Иванов купил ноутбук Lenovo ThinkPad X1 Carbon за 120000 рублей 15 января 2025 года.
Он оплатил картой Visa и выбрал доставку курьером на дом по адресу ул. Ленина, 45, кв. 12, Москва.
Номер заказа: ORD-2025-00123. Email: alex.ivanov@email.com, телефон: +7-915-123-4567.
"""

# System prompt для извлечения данных
extraction_prompt = """
Извлеки информацию о заказе из текста и верни ТОЛЬКО JSON в следующем формате:
{
    "customer_name": "имя клиента",
    "product": "название товара",
    "price": цена (число),
    "date": "дата покупки",
    "order_id": "номер заказа",
    "email": "email",
    "phone": "телефон"
}

Верни ТОЛЬКО JSON, без дополнительного текста.
"""

### Эксперимент 1: Temperature = 0.0 (детерминированный)

In [ ]:
def extract_data(text, temperature):
    """
    Извлечение данных из текста с заданной температурой
    Использует GPT-5 mini
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": extraction_prompt},
            {"role": "user", "content": text}
        ],
        temperature=temperature,
        max_completion_tokens=2000
    )
    return response.choices[0].message.content

# Запуск 3 раза с temperature = 0.0
print("=== Temperature = 0.0 (детерминированный) ===")
print("\n--- Попытка 1 ---")
result1_t0 = extract_data(text_to_parse, temperature=0.0)
print(result1_t0)

print("\n--- Попытка 2 ---")
result2_t0 = extract_data(text_to_parse, temperature=0.0)
print(result2_t0)

print("\n--- Попытка 3 ---")
result3_t0 = extract_data(text_to_parse, temperature=0.0)
print(result3_t0)

# Проверка идентичности
print("\n=== Анализ ===")
print(f"Результат 1 == Результат 2: {result1_t0 == result2_t0}")
print(f"Результат 2 == Результат 3: {result2_t0 == result3_t0}")
print(f"Все результаты идентичны: {result1_t0 == result2_t0 == result3_t0}")

**Ожидаемый результат:** Все 3 результата должны быть **идентичны** ✅

### Эксперимент 2: Temperature = 1.5 (высокая креативность)

In [ ]:
# Запуск 3 раза с temperature = 1.5
print("=== Temperature = 1.5 (высокая креативность) ===")
print("\n--- Попытка 1 ---")
result1_t15 = extract_data(text_to_parse, temperature=1.5)
print(result1_t15)

print("\n--- Попытка 2 ---")
result2_t15 = extract_data(text_to_parse, temperature=1.5)
print(result2_t15)

print("\n--- Попытка 3 ---")
result3_t15 = extract_data(text_to_parse, temperature=1.5)
print(result3_t15)

# Проверка различий
print("\n=== Анализ ===")
print(f"Результат 1 == Результат 2: {result1_t15 == result2_t15}")
print(f"Результат 2 == Результат 3: {result2_t15 == result3_t15}")
print(f"Все результаты идентичны: {result1_t15 == result2_t15 == result3_t15}")

**Ожидаемый результат:** Результаты **различаются**, могут появиться:
- Дополнительные поля в JSON
- Лишний текст вне JSON
- Креативные интерпретации
- Ошибки форматирования ❌

### Попытка парсинга JSON

In [ ]:
# Попытка распарсить JSON результаты
def try_parse_json(text, label):
    try:
        parsed = json.loads(text)
        print(f"✅ {label}: JSON валидный")
        return parsed
    except json.JSONDecodeError as e:
        print(f"❌ {label}: Ошибка парсинга JSON - {e}")
        return None

print("=== Парсинг результатов Temperature = 0.0 ===")
try_parse_json(result1_t0, "Попытка 1 (t=0.0)")
try_parse_json(result2_t0, "Попытка 2 (t=0.0)")
try_parse_json(result3_t0, "Попытка 3 (t=0.0)")

print("\n=== Парсинг результатов Temperature = 1.5 ===")
try_parse_json(result1_t15, "Попытка 1 (t=1.5)")
try_parse_json(result2_t15, "Попытка 2 (t=1.5)")
try_parse_json(result3_t15, "Попытка 3 (t=1.5)")

---

# Задание 2.2: Выводы о температуре

**Ответьте на вопросы:**

1. Какая температура подходит для извлечения структурированных данных?
2. Когда использовать высокую температуру?
3. Что происходит с форматом при высокой температуре?

### Ваши ответы:

**Вопрос 1: Какую температуру использовать для извлечения JSON?**

_Ваш ответ:_


**Вопрос 2: Когда использовать высокую температуру (1.0-1.5)?**

_Ваш ответ:_


**Вопрос 3: Какие проблемы возникают при высокой температуре?**

_Ваш ответ:_

---

# Рекомендации по температуре

| Задача | Температура | Причина |
|--------|-------------|----------|
| Извлечение JSON | 0.0 - 0.2 | Нужна точность и стабильность формата |
| Генерация кода | 0.0 - 0.3 | Код должен быть корректным |
| Чат-бот | 0.7 - 0.9 | Баланс точности и естественности |
| Креативное письмо | 1.0 - 1.5 | Нужно разнообразие и оригинальность |
| Brainstorming | 1.2 - 1.5 | Максимум различных идей |

---

# Дополнительно: GPT-5 Features

GPT-5 поддерживает новые параметры:

## 1. Reasoning Effort (усилие рассуждения)
```python
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[...],
    reasoning_effort="medium"  # minimal, low, medium, high
)
```

## 2. Verbosity (многословность)
```python
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[...],
    verbosity="low"  # low, medium, high
)
```

In [ ]:
# Пример использования новых параметров GPT-5
# ВНИМАНИЕ: Эти параметры могут быть не доступны в ранних версиях API

def chat_with_gpt5_features(user_message, reasoning_effort="medium", verbosity="medium"):
    """
    Демонстрация новых возможностей GPT-5
    """
    try:
        response = client.chat.completions.create(
            model="gpt-5-mini",
            messages=[
                {"role": "system", "content": "Ты полезный ассистент"},
                {"role": "user", "content": user_message}
            ],
            temperature=0.7,
            # Новые параметры GPT-5 (если доступны)
            # reasoning_effort=reasoning_effort,
            # verbosity=verbosity
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Ошибка: {e}"

# Тест
# result = chat_with_gpt5_features("Объясни квантовую физику")
# print(result)

---

# Итоги

В этом ноутбуке вы:

✅ Узнали как использовать **GPT-5 mini** через OpenAI API

✅ Создали профессиональный **system prompt** для чат-бота

✅ Экспериментировали с параметром **temperature**

✅ Поняли, когда использовать низкую/высокую температуру

## Следующие шаги:

- Задание 3.1: Реализация памяти (Conversation History)
- Финальный проект: TechSupportBot

---

**Документация:**
- [GPT-5 Models](https://platform.openai.com/docs/models/gpt-5)
- [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
- [GPT-5 Guide](https://platform.openai.com/docs/guides/latest-model)